# Comparative Sequence, Physicochemical, and Phylogenetic Analysis of Mpox (Monkeypox Virus) Surface Proteins A29L and A35R Across Clade I and Clade II

| | |
|---|---|
| **Institution** | Strathmore University |
| **Course** | Computing and Bioinformatics |
| **Author** | Cindy Ogutu |
| **Email** | cindy.ogutu@strathmore.edu |
| **Tooling** | Python 3, Biopython, pandas, NumPy, SciPy, Matplotlib, Seaborn |

---

## 1. Header, Research Objectives & KDD Workflow Overview

### 1.1 Biological background

| Protein | Length | Vaccinia ortholog | Role |
|---|---|---|---|
| **A29L** | ~110 aa (canonical filter `95:125[SLEN]`) | A27L | Intracellular Mature Virus (IMV) surface-binding protein. Mediates attachment to host-cell heparan sulfate and takes part in viral fusion. |
| **A35R** | ~181 aa (canonical filter `165:195[SLEN]`) | A33R | Extracellular Enveloped Virus (EEV) envelope glycoprotein. Needed for actin-tail formation and cell-to-cell spread, and a target of humoral immunity. |

**Clades compared**

* **Clade I** (formerly Central African / Congo Basin clade): historically higher virulence and case-fatality rate.
* **Clade II** (formerly West African clade), including **Clade IIb**, the lineage behind the 2022 global outbreak.

Both proteins are exposed on virions and are antigens in subunit and mRNA vaccine designs. Mutations that distinguish the clades could therefore change surface chemistry, antigenic conservation or the stability of diagnostic and vaccine targets.

### 1.2 Research objectives

1. Retrieve and curate all available Mpox A29L and A35R protein sequences reproducibly (local-first, with an API fallback).
2. Classify each record as **Clade I**, **Clade II** or **Unclassified** from GenBank metadata, and record the evidence used.
3. Identify clade-defining residues and rank positions by Shannon entropy.
4. Test whether clades differ in MW, pI, GRAVY, instability index, aromaticity and net charge (Mann-Whitney U).
5. Reconstruct a similarity tree (UPGMA hierarchical clustering) and check whether the clades separate.
6. Discuss what the results imply for pan-Mpox vaccines and diagnostics.

### 1.3 Project mapped to the KDD workflow

```
 KDD phase                  Notebook section   What happens here
 -------------------------  -----------------  -----------------------------------------------------
 +-----------------------+
 | 1. SELECTION          |  Section 2          3-tier retrieval: local GenBank export -> cache -> Entrez
 +-----------+-----------+
             v
 +-----------------------+
 | 2. PREPROCESSING      |  Section 3          dedup by accession, canonical length filter,
 |    & CLEANING         |                     remove ambiguous residues (X/B/Z/*)
 +-----------+-----------+
             v
 +-----------------------+
 | 3. TRANSFORMATION     |  Section 3 + 5      parse /isolate /strain /country /collection_date /note,
 |                       |                     classify clade, compute physicochemical feature vectors
 +-----------+-----------+
             v
 +-----------------------+
 | 4. DATA MINING /      |  Section 4 + 6      global + local alignment, conservation profile,
 |    SEQUENCE ANALYSIS  |                     Shannon entropy, distance matrix, hierarchical clustering
 +-----------+-----------+
             v
 +-----------------------+
 | 5. INTERPRETATION &   |  Section 5 + 7      clade separation, mutation hotspots, antigenic stability,
 |    EVALUATION         |                     limitations and reproducibility
 +-----------------------+
```

### 1.4 How to supply the data (Tier 1, recommended)

Local files give instant, reproducible parsing with no API limits. The notebook looks for these two files in the working directory first:

| File | Protein | NCBI Protein search (paste into the search box) |
|---|---|---|
| `mpox_a29l.gb` | A29L | `("Monkeypox virus"[Organism] OR "Mpox virus"[Organism]) AND ("A29L" OR "A29" OR "surface-binding") AND 95:125[SLEN]` |
| `mpox_a35r.gb` | A35R | `("Monkeypox virus"[Organism] OR "Mpox virus"[Organism]) AND ("A35R" OR "A35" OR "A33R" OR "A33" OR "EEV glycoprotein") AND 165:195[SLEN]` |

**Steps (do this once per protein)**

1. Open <https://www.ncbi.nlm.nih.gov/protein/> and run the search above.
2. Click **Send to** (top right of the results list) and choose **File**.
3. Set **Format** to the full GenBank flat file. For the Protein database NCBI labels this **GenPept (full)** (some views say *GenBank (full)*). Biopython reads it with `format="genbank"`.
4. Click **Create File** and save it as `mpox_a29l.gb` / `mpox_a35r.gb` in the same folder as this notebook. In Google Colab, upload them through the **Files** pane.

The A35R search deliberately omits the generic term `envelope`, which also matches unrelated envelope proteins of similar length.

If the files are absent, Tier 2 (`mpox_raw_gb_cache.gb`) is tried next, and finally Tier 3 (chunked Entrez download, which then writes that cache).

## 2. Environment Setup & 3-Tier Data Retrieval  *(KDD phase 1: Selection)*

`socket.setdefaulttimeout(15)` is the first executable line, so no HTTP request can hang indefinitely (the original `Entrez.efetch` hang).

In [ ]:
# ----------------------------------------------------------------------
# Anti-hang safety net: NO network call may block for more than 15 seconds.
# ----------------------------------------------------------------------
import socket
socket.setdefaulttimeout(15)

import io, os, re, sys, json, time, hashlib, warnings, subprocess
import http.client
from collections import Counter, OrderedDict
from datetime import datetime
from urllib.error import HTTPError, URLError

try:
    import Bio
except ImportError:                                   # fresh Colab / venv
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "biopython"])
    import Bio

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

from Bio import Entrez, SeqIO, Align
from Bio.Align import substitution_matrices
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParamData import kd as KYTE_DOOLITTLE

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---------------- Project constants ----------------
Entrez.email = "cindy.ogutu@strathmore.edu"
Entrez.tool = "Strathmore_Mpox_Bioinformatics_Project"
Entrez.max_tries = 1              # our own retry loop (below) controls retries, so a dead network fails fast
Entrez.sleep_between_tries = 2

PROTEINS = ["A29L", "A35R"]
LOCAL_FILES = {"A29L": "mpox_a29l.gb", "A35R": "mpox_a35r.gb"}       # Tier 1
CACHE_FILE = "mpox_raw_gb_cache.gb"                                   # Tier 2
CACHE_HINTS = "mpox_raw_gb_cache.hints.json"                          # protein tag per accession
LENGTH_RANGE = {"A29L": (95, 125), "A35R": (165, 195)}                # canonical filters

QUERIES = [                                                           # Tier 3 (exact 4 queries)
    ("A29L", '("Monkeypox virus"[Organism] OR "Mpox virus"[Organism]) AND ("A29L" OR "A29" OR "surface-binding") AND 95:125[SLEN] AND ("Clade I" OR "Clade 1" OR "Congo" OR "Central African")'),
    ("A29L", '("Monkeypox virus"[Organism] OR "Mpox virus"[Organism]) AND ("A29L" OR "A29" OR "surface-binding") AND 95:125[SLEN]'),
    ("A35R", '("Monkeypox virus"[Organism] OR "Mpox virus"[Organism]) AND ("A35R" OR "A35" OR "A33R" OR "A33" OR "EEV glycoprotein") AND 165:195[SLEN] AND ("Clade I" OR "Clade 1" OR "Congo" OR "Central African")'),
    ("A35R", '("Monkeypox virus"[Organism] OR "Mpox virus"[Organism]) AND ("A35R" OR "A35" OR "A33R" OR "A33" OR "EEV glycoprotein") AND 165:195[SLEN]'),
]
MAX_PER_QUERY, CHUNK_SIZE, SLEEP_S, MAX_RETRIES = 150, 50, 0.4, 3

CLADES = ["Clade I", "Clade II"]
CLADE_COLORS = {"Clade I": "#D55E00", "Clade II": "#0072B2", "Unclassified": "#999999"}   # colour-blind safe
os.makedirs("figures", exist_ok=True)


def savefig(fig, name):
    fig.savefig(os.path.join("figures", name), bbox_inches="tight")


def _log(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)


# ---------------- Protein identification helpers ----------------
PROT_PATTERNS = {
    "A29L": re.compile(r"\bA29L?\b|\bA27L?\b|surface[- ]binding|intracellular mature|\bIMV\b", re.I),
    "A35R": re.compile(r"\bA35R?\b|\bA33R?\b|\bEEV[\s-]+(?:membrane\s+|envelope\s+)?glycoprotein", re.I),
}


def record_text(rec):
    """Description + the descriptive qualifiers of every feature, as one string."""
    parts = [rec.description or ""]
    for f in rec.features:
        for k in ("gene", "product", "note", "name", "function", "strain", "isolate"):
            parts.extend(f.qualifiers.get(k, []))
    return " | ".join(parts)


def infer_protein(rec):
    """Protein tag: query/file hint first, else keywords (+ canonical length) in the annotation."""
    hint = rec.annotations.get("protein_hint")
    if hint in PROTEINS:
        return hint
    txt, n = record_text(rec), len(rec.seq)
    cands = [p for p, pat in PROT_PATTERNS.items()
             if pat.search(txt) and LENGTH_RANGE[p][0] <= n <= LENGTH_RANGE[p][1]]
    return cands[0] if len(cands) == 1 else None


# ---------------- Fault-tolerant network helpers (Tier 3) ----------------
NET_ERRORS = (socket.timeout, TimeoutError, HTTPError, URLError, ConnectionError,
              http.client.HTTPException, OSError, RuntimeError, ValueError)


def _retry(fn, what):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn()
        except NET_ERRORS as e:
            _log(f"   ! {what}: attempt {attempt}/{MAX_RETRIES} failed ({type(e).__name__}: {e})")
            time.sleep(1.5 * attempt)
    _log(f"   x {what}: giving up after {MAX_RETRIES} attempts")
    return None


def _esearch(query, retmax):
    def _do():
        h = Entrez.esearch(db="protein", term=query, retmax=retmax)
        try:
            return Entrez.read(h)["IdList"]
        finally:
            h.close()
    return _retry(_do, "esearch")          # None = network unreachable after all retries


def _efetch_chunk(ids):
    def _do():
        h = Entrez.efetch(db="protein", id=",".join(ids), rettype="gb", retmode="text")
        try:
            txt = h.read()
        finally:
            h.close()
        if isinstance(txt, bytes):
            txt = txt.decode("utf-8", "replace")
        recs = list(SeqIO.parse(io.StringIO(txt), "genbank"))
        if not recs:
            raise ValueError("empty or invalid GenBank payload")
        return recs
    return _retry(_do, f"efetch({len(ids)} ids)")


def _parse_gb(path):
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        return list(SeqIO.parse(fh, "genbank"))


def get_mpox_sequences_robust():
    """3-tier, local-first retrieval. Returns (list of SeqRecord, summary DataFrame).

    Tier 1: mpox_a29l.gb / mpox_a35r.gb   (manual NCBI export, instant)
    Tier 2: mpox_raw_gb_cache.gb          (cache written by an earlier Tier 3 run)
    Tier 3: chunked Entrez efetch         (50 IDs/chunk, retries, dedup, writes cache)
    Tiers 2 and 3 are only used for proteins that Tier 1 did not supply.
    """
    records, covered = OrderedDict(), set()

    # ---- Tier 1 ----
    _log("Tier 1: looking for manually exported local files ...")
    for prot, fn in LOCAL_FILES.items():
        if os.path.isfile(fn) and os.path.getsize(fn) > 0:
            try:
                recs = _parse_gb(fn)
            except ValueError as e:
                _log(f"   x {fn}: could not be parsed ({e})")
                continue
            new = 0
            for r in recs:
                if r.id not in records:
                    r.annotations["protein_hint"] = prot
                    r.annotations["data_tier"] = "Tier 1: local export"
                    records[r.id] = r
                    new += 1
            covered.add(prot)
            _log(f"   OK {fn}: {len(recs)} records parsed ({new} new)")
        else:
            _log(f"   -  {fn} not found")

    # ---- Tier 2 ----
    cache_pool, hints = OrderedDict(), {}
    missing = [p for p in PROTEINS if p not in covered]
    if missing:
        _log(f"Tier 2: checking local cache for {missing} ...")
        if os.path.isfile(CACHE_FILE) and os.path.getsize(CACHE_FILE) > 0:
            if os.path.isfile(CACHE_HINTS):
                try:
                    hints = json.load(open(CACHE_HINTS))
                except ValueError:
                    hints = {}
            try:
                for r in _parse_gb(CACHE_FILE):
                    cache_pool[r.id] = r
            except ValueError as e:
                _log(f"   x cache unreadable ({e})")
            for rid, r in cache_pool.items():
                r.annotations["protein_hint"] = hints.get(rid)
                prot = infer_protein(r)
                if prot in missing and rid not in records:
                    r.annotations["protein_hint"] = prot
                    r.annotations["data_tier"] = "Tier 2: local cache"
                    records[rid] = r
            for p in missing:
                n = sum(1 for r in records.values()
                        if r.annotations.get("protein_hint") == p
                        and r.annotations.get("data_tier") == "Tier 2: local cache")
                if n:
                    covered.add(p)
                    _log(f"   OK {p}: {n} records from {CACHE_FILE}")
        else:
            _log(f"   -  {CACHE_FILE} not found or empty")

    # ---- Tier 3 ----
    missing = [p for p in PROTEINS if p not in covered]
    if missing:
        _log(f"Tier 3: Entrez API fallback for {missing} (timeout 15 s, {MAX_RETRIES} retries, {CHUNK_SIZE}-ID chunks)")
        ids_by_prot = {p: OrderedDict() for p in missing}
        for prot, q in QUERIES:
            if prot not in missing:
                continue
            ids = _esearch(q, MAX_PER_QUERY)
            if ids is None:
                _log("   NCBI unreachable - aborting Tier 3 (use Tier 1 local files, see Section 1.4)")
                break
            new = [i for i in ids if i not in ids_by_prot[prot]]
            for i in new:
                ids_by_prot[prot][i] = True
            _log(f"   esearch [{prot}] -> {len(ids)} IDs ({len(new)} new)")
            time.sleep(SLEEP_S)

        fetched = OrderedDict()
        for prot in missing:
            ids = list(ids_by_prot[prot])
            chunks = [ids[i:i + CHUNK_SIZE] for i in range(0, len(ids), CHUNK_SIZE)]
            for k, ch in enumerate(chunks, 1):
                recs = _efetch_chunk(ch)
                if recs is None:
                    _log(f"   chunk {k}/{len(chunks)} [{prot}] skipped")
                    continue
                added = 0
                for r in recs:
                    if r.id not in records:
                        r.annotations["protein_hint"] = prot
                        r.annotations["data_tier"] = "Tier 3: Entrez API"
                        records[r.id] = r
                        fetched[r.id] = r
                        added += 1
                _log(f"   chunk {k}/{len(chunks)} [{prot}]: +{added} records (running total {len(records)})")
                time.sleep(SLEEP_S)

        if fetched:                                    # freeze to disk (merge with any old cache)
            merged = OrderedDict(cache_pool)
            merged.update(fetched)
            new_hints = dict(hints)
            new_hints.update({rid: r.annotations["protein_hint"] for rid, r in fetched.items()})
            try:
                SeqIO.write(list(merged.values()), CACHE_FILE, "genbank")
                json.dump(new_hints, open(CACHE_HINTS, "w"))
                _log(f"   saved {len(merged)} records to {CACHE_FILE}")
            except (ValueError, OSError) as e:
                _log(f"   ! could not write cache: {e}")

    if not records:
        raise RuntimeError(
            "No sequences could be loaded. Download mpox_a29l.gb and mpox_a35r.gb from the NCBI "
            "Protein web portal (see Section 1.4) and place them next to this notebook.")

    summary = (pd.DataFrame({"Protein": [r.annotations.get("protein_hint") or "?" for r in records.values()],
                             "Source": [r.annotations["data_tier"] for r in records.values()]})
               .value_counts().rename("Records").reset_index())
    return list(records.values()), summary


RAW_RECORDS, RETRIEVAL_SUMMARY = get_mpox_sequences_robust()
print(f"\nRaw records loaded: {len(RAW_RECORDS)}")
display(RETRIEVAL_SUMMARY)

## 3. Metadata Extraction, QC & Clade Classification  *(KDD phases 2-3: Preprocessing & Transformation)*

**Clade classification rules, applied in priority order.** All free text is first normalised (`_` and `/` become spaces) so that tokens such as `Yambuku_DRC_1985` match word-boundary regexes. The rule that fired is stored in `Clade_Evidence`.

1. **Explicit label** in description, isolate, strain, note or comment (e.g. "clade Ib", "Clade IIb").
2. **Lineage nomenclature** of the 2022 outbreak (`B.1`, `hMPXV-1`, ...), which is Clade IIb.
3. **Keywords** in isolate, strain, note, description or the `/coded_by` nucleotide accession. Clade I: Zaire, Congo, DRC, Yambuku, Boende, Kinshasa, Katanga, Sankuru and similar. Clade II: West Africa, Nigeria, Sierra Leone, Liberia, Ghana, hMpxV, `MPXV USA`, `MPXV Human`, `2022/NL`, `2022/PT`, `2022/UK` and similar, and the outbreak nucleotide prefixes `ON563` and `MT903`. Conflicting keywords leave the record Unclassified.
4. **Country of origin.** Clade I: DR Congo, Republic of the Congo, Central African Republic, Cameroon, Sudan, South Sudan, Gabon. Clade II: West African countries. Uganda, Rwanda, Burundi and Kenya count as Clade I only for collection year 2024 or later (clade Ib spread).
5. **Year rule.** A record collected in 2022 or 2023 that matched no Clade I marker or country is Clade II, **even when the country is unknown** (virtually all such sequences belong to the 2022 global IIb outbreak).

**Metadata sources.** NCBI Protein (GenPept) records often keep `/country` and `/collection_date` only on the linked *nucleotide* record, which the protein's `/coded_by` qualifier points to. The pipeline therefore extracts every available signal from `isolate`, `strain`, `note`, the description, the collection year and the `/coded_by` accession (stored in `Coded_By`). Records with no usable signal stay Unclassified.

**Ortholog identity QC.** After protein assignment, each sequence is globally aligned to its protein's modal (most frequent) sequence. Sequences below 75% identity are non-orthologous proteins that matched the search terms, and they are dropped and counted in the QC funnel. This also cleans any pre-existing `mpox_raw_gb_cache.gb`.

**Caveat.** Cameroon hosts both clades. Rule 4 follows the project brief, so treat Cameroon-only calls as lower confidence (check `Clade_Evidence`).

In [ ]:
# ----------------------- Metadata parsing & clade rules -----------------------
CLADE_TXT = re.compile(r"\bclade[\s_\-:]*(iib|iia|ib|ia|ii|i|2b|2a|1b|1a|2|1)\b", re.I)
LINEAGE_II = re.compile(r"\bB\.1(?:\.\d+)*\b|\bhMPXV[-_ ]?1\b|\blineage[\s_]+[ABC]\.\d", re.I)
KW_I = re.compile(r"zaire|congo|\bdrc\b|yambuku|boende|katanga|kinshasa|sankuru|tshuapa|equateur|uele|ubangi|kwilu|tshopo|"
                  r"central africa|congo basin", re.I)
KW_II = re.compile(r"west africa|nigeria|sierra leone|liberia|ghana|ivory coast|cote d.ivoire|c.te d.ivoire|"
                   r"benin|hmpxv|MPXV[- ]?UK[- ]P\d+|MPXV[- ]USA|MPXV[- ]Human|"
                   r"\b2022[- ](?:NL|PT|UK|ES|DE|FR|IT|BE|CA|US|AU|CH|SE|DK|IE|AT)\b|\bON563\d*|\bMT903\d*|"
                   r"\bMA001\b|rivers|bayelsa", re.I)

COUNTRY_ALIASES = {
    "drc": "Democratic Republic of the Congo", "dr congo": "Democratic Republic of the Congo",
    "democratic republic of congo": "Democratic Republic of the Congo",
    "democratic republic of the congo": "Democratic Republic of the Congo",
    "congo, democratic republic of the": "Democratic Republic of the Congo",
    "zaire": "Democratic Republic of the Congo", "republic of congo": "Republic of the Congo",
    "republic of the congo": "Republic of the Congo", "car": "Central African Republic",
    "usa": "USA", "united states": "USA", "uk": "United Kingdom", "great britain": "United Kingdom",
    "england": "United Kingdom", "cote d'ivoire": "Cote d'Ivoire", "côte d'ivoire": "Cote d'Ivoire",
    "ivory coast": "Cote d'Ivoire",
}
CLADE_I_COUNTRIES = {"democratic republic of the congo", "republic of the congo", "congo",
                     "central african republic", "cameroon", "sudan", "south sudan", "gabon"}
CLADE_I_RECENT = {"uganda", "rwanda", "burundi", "kenya"}                      # clade Ib, year >= 2024
CLADE_II_WEST_AFRICA = {"nigeria", "ghana", "sierra leone", "liberia", "cote d'ivoire", "benin", "togo", "guinea"}


def norm_country(raw):
    if not raw:
        return "Unknown"
    c = raw.split(":")[0].strip()
    return COUNTRY_ALIASES.get(c.lower(), c)


def parse_year(*texts):
    for t in texts:
        m = re.search(r"\b(19[5-9]\d|20[0-2]\d)\b", t or "")
        if m:
            return int(m.group(1))
    return None


def parse_source(rec):
    q = {}
    for f in rec.features:
        if f.type == "source":
            q = {k: "; ".join(v) for k, v in f.qualifiers.items()}
            break
    return q


def classify_clade(free_text, country, year):
    free_text = re.sub(r"[_/]+", " ", free_text)        # '_' is a regex word char: 'Yambuku_DRC_1985' broke \bdrc\b
    m = CLADE_TXT.search(free_text)
    if m:
        code_ = m.group(1).upper()
        clade = "Clade I" if code_ in {"I", "IA", "IB", "1", "1A", "1B"} else "Clade II"
        return clade, f"explicit label: clade {m.group(1)}"
    m = LINEAGE_II.search(free_text)
    if m:
        return "Clade II", f"lineage marker: {m.group(0)}"
    hi, hii = KW_I.search(free_text), KW_II.search(free_text)
    if hi and not hii:
        return "Clade I", f"keyword: {hi.group(0)}"
    if hii and not hi:
        return "Clade II", f"keyword: {hii.group(0)}"
    if hi and hii:
        return "Unclassified", "conflicting keywords"
    c = country.lower()
    if c in CLADE_I_COUNTRIES:
        return "Clade I", f"country: {country}"
    if c in CLADE_I_RECENT and year is not None and year >= 2024:
        return "Clade I", f"country+year: {country} {year} (clade Ib)"
    if c in CLADE_II_WEST_AFRICA:
        return "Clade II", f"country: {country} (West Africa)"
    if year in (2022, 2023) and c not in CLADE_I_COUNTRIES:      # country often missing on protein records
        return "Clade II", f"year rule: {year} global outbreak (IIb)"
    return "Unclassified", "insufficient metadata"


def record_to_row(rec):
    src = parse_source(rec)
    seq = str(rec.seq).upper()
    country = norm_country(src.get("country", ""))
    isolate = src.get("isolate") or src.get("strain") or ""
    coded_by = ""                                              # link to the nucleotide record, e.g. ON563414.3:1..545
    for f in rec.features:
        if f.type == "CDS" and f.qualifiers.get("coded_by"):
            coded_by = "; ".join(f.qualifiers["coded_by"])
            break
    year = parse_year(src.get("collection_date"), src.get("isolate"), src.get("strain"), rec.description)
    free = " | ".join([rec.description or "", src.get("isolate", ""), src.get("strain", ""),
                       src.get("note", ""), coded_by, str(rec.annotations.get("comment", ""))])
    clade, evidence = classify_clade(free, country, year)
    prot = infer_protein(rec)
    txt_match = bool(prot and PROT_PATTERNS[prot].search(record_text(rec)))
    return {"Accession": rec.id, "Protein": prot or "Unassigned", "Clade": clade, "Clade_Evidence": evidence,
            "Country": country, "Year": year, "Isolate": isolate, "Length": len(seq), "Sequence": seq,
            "Description": rec.description, "Coded_By": coded_by, "Annot_Match": txt_match,
            "Data_Tier": rec.annotations.get("data_tier", "")}


df_parsed = pd.DataFrame([record_to_row(r) for r in RAW_RECORDS])
df_parsed["Year"] = df_parsed["Year"].astype("Int64")
print(f"Parsed {len(df_parsed)} records into a DataFrame.")
display(df_parsed.drop(columns="Sequence").head(8))

In [ ]:
# ----------------------- Quality control funnel -----------------------
qc = []
df = df_parsed.copy()
qc.append(("Raw parsed records", len(df)))

df = df[df.Protein.isin(PROTEINS)].copy()
qc.append(("Assigned to A29L / A35R", len(df)))

# de-duplicate by accession (ignore version number, keep the highest version)
df["_base"] = df.Accession.str.split(".").str[0]
df["_ver"] = pd.to_numeric(df.Accession.str.split(".").str[1], errors="coerce").fillna(1)
df = df.sort_values(["_base", "_ver"]).drop_duplicates("_base", keep="last").drop(columns=["_base", "_ver"])
qc.append(("After accession de-duplication", len(df)))

in_range = df.apply(lambda r: LENGTH_RANGE[r.Protein][0] <= r.Length <= LENGTH_RANGE[r.Protein][1], axis=1)
df = df[in_range].copy()
qc.append(("Canonical length filter (A29L 95-125, A35R 165-195)", len(df)))

# any residue outside the 20 standard amino acids (X, B, Z, *, J, U, O ...) is ambiguous
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
has_ambig = df.Sequence.apply(lambda s: not set(s) <= STANDARD_AA)
n_ambig = int(has_ambig.sum())
df = df[~has_ambig].copy()
qc.append((f"Removed ambiguous residues (X/B/Z/*...; {n_ambig} seqs)", len(df)))

# ---- Ortholog identity QC: drop non-orthologous proteins that slipped through broad search terms ----
ORTHO_MIN_ID = 0.75                       # minimum identity to the protein's modal reference
BLOSUM62 = substitution_matrices.load("BLOSUM62")


def make_aligner(mode="global"):
    a = Align.PairwiseAligner()
    a.mode = mode
    a.substitution_matrix = BLOSUM62
    a.open_gap_score = -10.0
    a.extend_gap_score = -0.5
    return a


def aln_stats(aln, ref, qry):
    """(matches, aligned columns, total alignment columns) from the aligned blocks."""
    blocks = list(zip(aln.aligned[0], aln.aligned[1]))
    aligned_cols = sum(e - s for (s, e), _ in blocks)
    matches = sum(sum(ref[i] == qry[j] for i, j in zip(range(rs, re_), range(qs, qe)))
                  for (rs, re_), (qs, qe) in blocks)
    return matches, aligned_cols, aligned_cols + (len(ref) - aligned_cols) + (len(qry) - aligned_cols)


_qc_aligner = make_aligner("global")


def pct_identity(ref, q):
    if ref == q:
        return 1.0
    m, _, cols = aln_stats(_qc_aligner.align(ref, q)[0], ref, q)
    return m / cols


bad_rows, n_ortho = [], 0
for prot in PROTEINS:
    g = df[df.Protein == prot]
    if g.empty:
        continue
    pool = g[g.Annot_Match] if g.Annot_Match.any() else g           # reference chosen among keyword-matched records
    ref = pool.Sequence.value_counts().idxmax()                     # modal sequence
    ident = g.Sequence.map({s: pct_identity(ref, s) for s in g.Sequence.unique()})
    bad = g[ident < ORTHO_MIN_ID]
    if len(bad):
        n_ortho += len(bad)
        bad_rows.append(bad.assign(Identity_to_modal=ident[bad.index].round(3))
                        .groupby(["Protein", "Length", "Identity_to_modal"])
                        .agg(n=("Accession", "size"), example=("Accession", "first"),
                             description=("Description", "first")).reset_index())
        df = df.drop(index=bad.index)
qc.append((f"Removed non-orthologous (<{ORTHO_MIN_ID:.0%} identity to modal reference; {n_ortho} seqs)", len(df)))
if bad_rows:
    print("Non-orthologous sequences removed:")
    display(pd.concat(bad_rows, ignore_index=True))

df = df.reset_index(drop=True)
display(pd.DataFrame(qc, columns=["QC step", "Records remaining"]))

n_unmatched = int((~df.Annot_Match).sum())
if n_unmatched:
    print(f"Note: {n_unmatched} records were assigned by file/query and length only (no A29/A35/A27/A33/envelope keyword "
          f"in the annotation). They are kept; inspect with df[~df.Annot_Match] if needed.")

df_cls = df[df.Clade.isin(CLADES)].copy()
print(f"\nClean dataset: {len(df)} sequences, of which {len(df_cls)} are clade-classified "
      f"({len(df) - len(df_cls)} Unclassified are excluded from clade comparisons).")
display(pd.crosstab(df.Protein, df.Clade, margins=True))
display(df.groupby(["Protein", "Clade", "Clade_Evidence"]).size().rename("n").reset_index()
        .sort_values("n", ascending=False).head(25))

In [ ]:
# ----------------------- Dataset composition plots -----------------------
clade_order = ["Clade I", "Clade II", "Unclassified"]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.2), gridspec_kw={"width_ratios": [1, 1, 1.9]})

ct = pd.crosstab(df.Protein, df.Clade).reindex(columns=clade_order, fill_value=0)       # (a) protein, stacked by clade
ct.plot(kind="bar", stacked=True, ax=axes[0], color=[CLADE_COLORS[c] for c in ct.columns], width=0.65, rot=0)
axes[0].set(title="(a) Sequences per protein, stacked by clade", xlabel="", ylabel="Number of sequences")

ct2 = pd.crosstab(df.Clade, df.Protein).reindex(clade_order, fill_value=0)              # (b) clade, stacked by protein
ct2.plot(kind="bar", stacked=True, ax=axes[1], color=["#009E73", "#CC79A7"], width=0.65, rot=0)
axes[1].set(title="(b) Sequences per clade, stacked by protein", xlabel="", ylabel="")

top = df.Country.value_counts().head(15).index                                         # (c) geography, stacked by clade
ct3 = (pd.crosstab(df[df.Country.isin(top)].Country, df[df.Country.isin(top)].Clade)
       .reindex(columns=clade_order, fill_value=0))
ct3 = ct3.loc[ct3.sum(axis=1).sort_values(ascending=False).index]
ct3.plot(kind="bar", stacked=True, ax=axes[2], color=[CLADE_COLORS[c] for c in ct3.columns], width=0.75)
axes[2].set(title="(c) Geographic origin (top 15 countries), stacked by clade", xlabel="", ylabel="")
axes[2].tick_params(axis="x", rotation=60)
for a in axes[2:]:
    for lbl in a.get_xticklabels():
        lbl.set_ha("right")
fig.tight_layout()
savefig(fig, "fig1_dataset_composition.png")
plt.show()

if df.Year.notna().any():
    fig, ax = plt.subplots(figsize=(9, 3.6))
    yr = pd.crosstab(df.Year.dropna().astype(int), df.Clade[df.Year.notna()]).reindex(columns=clade_order, fill_value=0)
    yr.plot(kind="bar", stacked=True, ax=ax, color=[CLADE_COLORS[c] for c in yr.columns], width=0.8)
    ax.set(title="Collection year (where recorded), stacked by clade", xlabel="Year", ylabel="Sequences")
    fig.tight_layout()
    savefig(fig, "fig2_year_composition.png")
    plt.show()

## 4. Global/Local Alignment & Clade-Defining Mutation Analysis  *(KDD phase 4: Data Mining)*

**Method**

1. **Haplotypes.** Identical sequences within a protein and clade collapse into one haplotype with a frequency count. IDs look like `A29L_I_H1` (the most frequent Clade I haplotype of A29L).
2. **Reference sequences.** The modal (most frequent) haplotype of each clade is its canonical reference.
3. **Pairwise alignment** with `Bio.Align.PairwiseAligner`: *global* (Needleman-Wunsch) and *local* (Smith-Waterman), BLOSUM62, gap open -10 and extend -0.5. This compares the Clade I reference with the Clade II reference.
4. **Multiple alignment (star alignment).** Every haplotype is globally aligned to the Clade I reference and projected into its coordinates. This is a sound approximation for near-identical orthologs. Insertions relative to the reference are dropped, and all numbering follows the Clade I reference.
5. **Shannon entropy** per column, $H = -\sum_i p_i \log_2 p_i$ over the 20 amino acids (gaps excluded, weighted by haplotype frequency). $H = 0$ means fully conserved and $H_{max}=\log_2 20 \approx 4.32$ bits.
6. **Mutation table.** Every variable position is listed. A substitution is *conservative* if its BLOSUM62 score is greater than 0, and *non-conservative* otherwise. The table also gives the change in net charge (K/R = +1, D/E = -1) and in Kyte-Doolittle hydropathy.

In [ ]:
# ----------------------- Haplotypes -----------------------
SHORT = {"Clade I": "I", "Clade II": "II"}


def _years(s):
    s = s.dropna()
    return "NA" if len(s) == 0 else (f"{int(s.min())}" if s.min() == s.max() else f"{int(s.min())}-{int(s.max())}")


hap_rows = []
for prot in PROTEINS:
    for clade in CLADES:
        g = df_cls[(df_cls.Protein == prot) & (df_cls.Clade == clade)]
        if g.empty:
            continue
        agg = (g.groupby("Sequence")
                 .agg(Freq=("Accession", "size"),
                      Accessions=("Accession", lambda s: ", ".join(list(s)[:3]) + (" ..." if len(s) > 3 else "")),
                      Countries=("Country", lambda s: ", ".join(sorted(set(s))[:4])),
                      Years=("Year", _years))
                 .reset_index().sort_values(["Freq", "Sequence"], ascending=[False, True]).reset_index(drop=True))
        for i, r in agg.iterrows():
            hap_rows.append({"Haplotype": f"{prot}_{SHORT[clade]}_H{i + 1}", "Protein": prot, "Clade": clade,
                             "Freq": int(r.Freq), "Rel_Freq_in_Clade": r.Freq / len(g), "Length": len(r.Sequence),
                             "Accessions": r.Accessions, "Countries": r.Countries, "Years": r.Years,
                             "Sequence": r.Sequence})
if not hap_rows:
    raise RuntimeError("No clade-classified sequences available. Check Clade_Evidence / metadata in Section 3.")
HAP = pd.DataFrame(hap_rows)
df_cls = df_cls.merge(HAP[["Protein", "Clade", "Sequence", "Haplotype"]], on=["Protein", "Clade", "Sequence"], how="left")

summ = (HAP.groupby(["Protein", "Clade"]).agg(Sequences=("Freq", "sum"), Unique_haplotypes=("Haplotype", "size"),
                                              Top_haplotype_share=("Rel_Freq_in_Clade", "max")).reset_index())
display(summ)
shared = HAP.groupby(["Protein", "Sequence"]).Clade.nunique()
print("Identical sequences shared by BOTH clades:", int((shared > 1).sum()), "(per protein: "
      + ", ".join(f"{p}={int((shared.loc[p] > 1).sum())}" for p in PROTEINS if p in shared.index.get_level_values(0)) + ")")
display(HAP.drop(columns="Sequence").head(15))

In [ ]:
# ----------------------- Pairwise global & local alignment of clade references -----------------------
BLOSUM62 = substitution_matrices.load("BLOSUM62")


def make_aligner(mode="global"):
    a = Align.PairwiseAligner()
    a.mode = mode
    a.substitution_matrix = BLOSUM62
    a.open_gap_score = -10.0
    a.extend_gap_score = -0.5
    return a


def aln_stats(aln, ref, qry):
    """Identity over alignment columns (global) from aligned blocks."""
    blocks = list(zip(aln.aligned[0], aln.aligned[1]))
    aligned_cols = sum(e - s for (s, e), _ in blocks)
    matches = sum(sum(ref[i] == qry[j] for i, j in zip(range(rs, re_), range(qs, qe)))
                  for (rs, re_), (qs, qe) in blocks)
    cols = aligned_cols + (len(ref) - aligned_cols) + (len(qry) - aligned_cols)
    return matches, aligned_cols, cols


REFS = {}                      # (protein, clade) -> (haplotype id, sequence)
for prot in PROTEINS:
    for clade in CLADES:
        g = HAP[(HAP.Protein == prot) & (HAP.Clade == clade)]
        if not g.empty:
            REFS[(prot, clade)] = (g.iloc[0].Haplotype, g.iloc[0].Sequence)

g_al, l_al = make_aligner("global"), make_aligner("local")
rows = []
for prot in PROTEINS:
    if (prot, "Clade I") not in REFS or (prot, "Clade II") not in REFS:
        print(f"{prot}: both a Clade I and a Clade II reference are required - skipping pairwise comparison.")
        continue
    (idI, sI), (idII, sII) = REFS[(prot, "Clade I")], REFS[(prot, "Clade II")]
    ga = g_al.align(sI, sII)[0]
    la = l_al.align(sI, sII)[0]
    m, ac, cols = aln_stats(ga, sI, sII)
    lm, lac, _ = aln_stats(la, sI, sII)
    rows.append({"Protein": prot, "Clade I ref": idI, "Clade II ref": idII,
                 "Global score": g_al.score(sI, sII), "Global identity %": 100 * m / cols,
                 "Global mismatches": ac - m, "Local score": l_al.score(sI, sII),
                 "Local aligned length": lac, "Local identity %": 100 * lm / max(lac, 1)})
    print(f"=== {prot}: global alignment of {idI} (target) vs {idII} (query) ===")
    print(ga)
if rows:
    display(pd.DataFrame(rows).round(2))

In [ ]:
# ----------------------- Star MSA, Shannon entropy, mutation table -----------------------
def project_to_reference(ref, seq, aligner):
    """Global-align seq to ref and return a string of len(ref) ('-' = deletion; insertions dropped)."""
    if seq == ref:
        return ref
    aln = aligner.align(ref, seq)[0]
    out = ["-"] * len(ref)
    for (rs, re_), (qs, qe) in zip(aln.aligned[0], aln.aligned[1]):
        out[rs:re_] = list(seq[qs:qe])
    return "".join(out)


def shannon(col, w):
    keep = col != "-"
    if not keep.any():
        return 0.0
    _, inv = np.unique(col[keep], return_inverse=True)
    p = np.bincount(inv, weights=w[keep])
    p = p / p.sum()
    return float(-(p * np.log2(p)).sum())


CHARGE = {"K": 1, "R": 1, "D": -1, "E": -1}
MUT_COLS = ["Protein", "Position", "Ref_residue", "Clade_I_residue", "Clade_I_freq", "Clade_II_residue",
            "Clade_II_freq", "All_variants", "Category", "Substitution", "Substitution_type", "BLOSUM62",
            "Delta_charge", "Delta_KD_hydropathy", "Entropy_bits"]
MAJ_FIXED = 0.90          # a clade's majority residue must reach this frequency to call a position "fixed"


def build_alignment(prot):
    haps = HAP[HAP.Protein == prot].reset_index(drop=True)
    ref_clade = "Clade I" if (prot, "Clade I") in REFS else "Clade II"
    ref_id, ref_seq = REFS[(prot, ref_clade)]
    M = np.array([list(project_to_reference(ref_seq, s, g_al)) for s in haps.Sequence])
    w = haps.Freq.values.astype(float)
    clades = haps.Clade.values
    L = M.shape[1]
    H_all = np.array([shannon(M[:, j], w) for j in range(L)])
    H_cl = {c: np.array([shannon(M[clades == c][:, j], w[clades == c]) for j in range(L)])
            for c in CLADES if (clades == c).any()}

    recs = []
    for j in range(L):
        col = M[:, j]
        if len(set(col)) < 2:
            continue
        info = {}
        for c in CLADES:
            msk = clades == c
            if not msk.any():
                continue
            tot = w[msk].sum()
            cnt = {}
            for aa, ww in zip(col[msk], w[msk]):
                cnt[aa] = cnt.get(aa, 0) + ww
            top, topn = max(cnt.items(), key=lambda kv: kv[1])
            info[c] = (top, topn / tot, cnt)
        allcnt = {}
        for aa, ww in zip(col, w):
            allcnt[aa] = allcnt.get(aa, 0) + ww
        a_I = info.get("Clade I", (None, np.nan, {}))
        a_II = info.get("Clade II", (None, np.nan, {}))
        if a_I[0] is not None and a_II[0] is not None and a_I[0] != a_II[0]:
            cat = "Clade-defining (fixed)" if min(a_I[1], a_II[1]) >= MAJ_FIXED else "Clade-biased (majority differs)"
            a, b = a_I[0], a_II[0]
        else:
            cat = "Intra-clade polymorphism"
            ranked = sorted(allcnt.items(), key=lambda kv: -kv[1])
            a, b = ranked[0][0], ranked[1][0]
        if a == "-" or b == "-":
            stype, bl, dq, dh = "Indel", np.nan, np.nan, np.nan
        else:
            bl = BLOSUM62[a][b]
            stype = "Conservative" if bl > 0 else "Non-conservative"
            dq = CHARGE.get(b, 0) - CHARGE.get(a, 0)
            dh = round(KYTE_DOOLITTLE[b] - KYTE_DOOLITTLE[a], 2)
        recs.append({"Protein": prot, "Position": j + 1, "Ref_residue": ref_seq[j],
                     "Clade_I_residue": a_I[0], "Clade_I_freq": round(a_I[1], 3) if a_I[0] else np.nan,
                     "Clade_II_residue": a_II[0], "Clade_II_freq": round(a_II[1], 3) if a_II[0] else np.nan,
                     "All_variants": ", ".join(f"{k}:{int(v)}" for k, v in sorted(allcnt.items(), key=lambda kv: -kv[1])),
                     "Category": cat, "Substitution": f"{a}->{b}", "Substitution_type": stype, "BLOSUM62": bl,
                     "Delta_charge": dq, "Delta_KD_hydropathy": dh, "Entropy_bits": round(H_all[j], 3)})
    return {"ref_id": ref_id, "ref_clade": ref_clade, "ref_seq": ref_seq, "haps": haps, "M": M, "w": w,
            "clades": clades, "H_all": H_all, "H_cl": H_cl, "mut": pd.DataFrame(recs, columns=MUT_COLS)}


ALN = {p: build_alignment(p) for p in PROTEINS if (HAP.Protein == p).any()}
for p, a in ALN.items():
    print(f"{p}: {a['M'].shape[0]} haplotypes aligned to {a['ref_id']} (length {a['M'].shape[1]}); "
          f"{len(a['mut'])} variable positions; {int((a['H_all'] > 0).sum())} columns with entropy > 0")

MUT = pd.concat([a["mut"] for a in ALN.values()], ignore_index=True) if ALN else pd.DataFrame()
print("\n=== Mutation summary table: every variable residue position ===")
display(MUT)
if len(MUT):
    display(MUT.groupby(["Protein", "Category", "Substitution_type"]).size().rename("positions").reset_index())
    MUT.to_csv("mpox_mutation_summary.csv", index=False)

In [ ]:
# ----------------------- Shannon entropy conservation profile -----------------------
fig, axes = plt.subplots(len(ALN), 1, figsize=(13, 3.9 * len(ALN)), squeeze=False)
for ax, (prot, a) in zip(axes[:, 0], ALN.items()):
    pos = np.arange(1, a["M"].shape[1] + 1)
    ax.fill_between(pos, a["H_all"], step="mid", color="#bbbbbb", alpha=0.6, label="All sequences")
    for c, h in a["H_cl"].items():
        ax.plot(pos, h, drawstyle="steps-mid", lw=1.6, color=CLADE_COLORS[c], label=f"{c} (within-clade)")
    m = a["mut"]
    cd = m[m.Category == "Clade-defining (fixed)"]
    if len(cd):
        ymax = max(a["H_all"].max(), 0.5)
        ax.scatter(cd.Position, [ymax * 1.08] * len(cd), marker="v", s=55, color="black", zorder=5,
                   label="Clade-defining position")
        for _, r in cd.iterrows():
            ax.annotate(r.Substitution, (r.Position, ymax * 1.08), xytext=(0, 7), textcoords="offset points",
                        ha="center", fontsize=7, rotation=90)
    ax.set(title=f"{prot} conservation profile (Shannon entropy; numbering follows {a['ref_id']})",
           xlabel="Residue position", ylabel="Entropy H (bits)", ylim=(0, max(a["H_all"].max(), 0.5) * 1.55))
    ax.legend(loc="upper left", ncol=4, fontsize=8, frameon=False)
fig.tight_layout()
savefig(fig, "fig3_entropy_profiles.png")
plt.show()

## 5. Physicochemical & Antigenic Property Profiling  *(KDD phases 3 and 5)*

`Bio.SeqUtils.ProtParam.ProteinAnalysis` gives, for every sequence:

| Feature | Meaning |
|---|---|
| **MW (Da)** | Molecular weight |
| **pI** | Isoelectric point (surface charge behaviour) |
| **GRAVY** | Grand Average of Hydropathicity (positive = hydrophobic) |
| **Instability index** | Values above 40 predict an unstable protein in vitro |
| **Aromaticity** | Fraction of F, W and Y |
| **Net charge at pH 7.4** | Extra feature (when supported by your Biopython version) |

Clade I and Clade II are compared with a **two-sided Mann-Whitney U test**. The rank-biserial correlation $r = 2U/(n_1 n_2) - 1$ is the effect size; $r>0$ means Clade I values tend to be larger. Because identical sequences are heavily duplicated in outbreak data, which inflates $n$, each test is also run on **unique haplotypes only**. Bonferroni correction is applied across all tests within each level.

In [ ]:
# ----------------------- Physicochemical features -----------------------
PROPS = ["MW_Da", "pI", "GRAVY", "Instability", "Aromaticity", "Charge_pH7.4"]
PROP_LABEL = {"MW_Da": "Molecular weight (Da)", "pI": "Isoelectric point (pI)", "GRAVY": "GRAVY",
              "Instability": "Instability index", "Aromaticity": "Aromaticity", "Charge_pH7.4": "Net charge, pH 7.4"}


def physchem(seq):
    pa = ProteinAnalysis(seq)
    d = {"MW_Da": pa.molecular_weight(), "pI": pa.isoelectric_point(), "GRAVY": pa.gravy(),
         "Instability": pa.instability_index(), "Aromaticity": pa.aromaticity()}
    try:
        d["Charge_pH7.4"] = pa.charge_at_pH(7.4)
    except Exception:
        d["Charge_pH7.4"] = np.nan
    return d


uniq = pd.Series(df.Sequence.unique())
feat = pd.DataFrame([physchem(s) for s in uniq]); feat.insert(0, "Sequence", uniq.values)
df_feat = df.merge(feat, on="Sequence", how="left")                  # all QC-passed sequences (incl. Unclassified)
df_feat_cls = df_feat[df_feat.Clade.isin(CLADES)].copy()
hap_feat = HAP.merge(feat, on="Sequence", how="left")                # one row per haplotype

print("Per-protein, per-clade summary (median [IQR]):")
agg = (df_feat.groupby(["Protein", "Clade"])[PROPS]
       .agg(lambda s: f"{s.median():.3f} [{s.quantile(.25):.3f}-{s.quantile(.75):.3f}]").reset_index())
display(agg)
df_feat.drop(columns="Sequence").to_csv("mpox_physicochemical_features.csv", index=False)

In [ ]:
# ----------------------- Mann-Whitney U tests -----------------------
def mwu_table(data, level):
    rows = []
    for prot in PROTEINS:
        for prop in PROPS:
            x = data[(data.Protein == prot) & (data.Clade == "Clade I")][prop].dropna().values
            y = data[(data.Protein == prot) & (data.Clade == "Clade II")][prop].dropna().values
            if len(x) < 2 or len(y) < 2:
                rows.append({"Level": level, "Protein": prot, "Property": prop, "n_I": len(x), "n_II": len(y)})
                continue
            try:
                U, p = stats.mannwhitneyu(x, y, alternative="two-sided")
            except ValueError:                       # all values identical
                U, p = len(x) * len(y) / 2, 1.0
            rows.append({"Level": level, "Protein": prot, "Property": prop, "n_I": len(x), "n_II": len(y),
                         "Median_I": np.median(x), "Median_II": np.median(y),
                         "U": U, "p_value": p, "rank_biserial_r": 2 * U / (len(x) * len(y)) - 1})
    t = pd.DataFrame(rows)
    if "p_value" in t:
        k = t.p_value.notna().sum()
        t["p_Bonferroni"] = (t.p_value * k).clip(upper=1.0)
        t["Significant (p<0.05)"] = t.p_Bonferroni < 0.05
    return t


MWU = pd.concat([mwu_table(df_feat_cls, "Sequence-level"), mwu_table(hap_feat, "Haplotype-level")], ignore_index=True)
display(MWU.round(5))
MWU.to_csv("mpox_mannwhitney_results.csv", index=False)
sig = MWU[MWU.get("Significant (p<0.05)", pd.Series(False, index=MWU.index)) == True]
print("\nProperties differing significantly between clades after Bonferroni correction:")
print(sig[["Level", "Protein", "Property", "Median_I", "Median_II", "p_Bonferroni"]].to_string(index=False) if len(sig)
      else "  none (clades are physicochemically indistinguishable for these global descriptors)")

In [ ]:
# ----------------------- Boxplots + stripplots -----------------------
def box_strip(ax, data, prop, title_p=None):
    groups = [data[data.Clade == c][prop].dropna().values for c in CLADES]
    bp = ax.boxplot(groups, positions=[0, 1], widths=0.5, patch_artist=True, showfliers=False)
    rng = np.random.default_rng(42)
    for i, c in enumerate(CLADES):
        bp["boxes"][i].set(facecolor=CLADE_COLORS[c], alpha=0.35, edgecolor=CLADE_COLORS[c])
        bp["medians"][i].set(color="black", lw=1.5)
        if len(groups[i]):
            ax.scatter(i + rng.uniform(-0.17, 0.17, len(groups[i])), groups[i], s=14, alpha=0.5,
                       color=CLADE_COLORS[c], edgecolor="none")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Clade I", "Clade II"])
    ax.set_ylabel(PROP_LABEL[prop]); ax.set_title(title_p or "", fontsize=9)


fig, axes = plt.subplots(len(PROTEINS), len(PROPS), figsize=(3.1 * len(PROPS), 3.6 * len(PROTEINS)), squeeze=False)
seq_lvl = MWU[MWU.Level == "Sequence-level"].set_index(["Protein", "Property"])
for i, prot in enumerate(PROTEINS):
    for j, prop in enumerate(PROPS):
        d = df_feat_cls[df_feat_cls.Protein == prot]
        try:
            pv = seq_lvl.loc[(prot, prop), "p_value"]
            ttl = f"{prot}  p = {pv:.3g}" if pd.notna(pv) else f"{prot}  (test n/a)"
        except KeyError:
            ttl = prot
        box_strip(axes[i, j], d, prop, ttl)
fig.suptitle("Physicochemical properties of A29L and A35R: Clade I vs Clade II (Mann-Whitney U, uncorrected p)", y=1.01)
fig.tight_layout()
savefig(fig, "fig4_physicochemical_boxplots.png")
plt.show()

## 6. Pairwise Distance Matrix & Hierarchical Clustering (Phylogeny)  *(KDD phase 4)*

**Distance.** For two aligned haplotypes (star-MSA columns), the identity is
$\text{identity} = \dfrac{\#\text{matching residues}}{\#\text{columns where at least one sequence has a residue}}$ and the normalized distance is $d = 1 - \text{identity}$.

**Tree.** Average-linkage hierarchical clustering (**UPGMA**, `scipy.cluster.hierarchy.linkage(method="average")`) on the representative haplotypes. Leaves are colour-coded by clade. At most `MAX_LEAVES` of the most frequent haplotypes per protein are shown to keep the tree legible.

> **Why hierarchical clustering rather than K-means?**
> * Evolutionary relationships are **nested** (a tree), not a flat partition. Hierarchical clustering returns a dendrogram with branch heights equal to dissimilarity. K-means returns only $k$ unrelated groups.
> * K-means needs **vectors in Euclidean space**, since it averages them into centroids, and the number of clusters $k$ in advance. Protein sequences give a **distance matrix** (a mean of sequences is not a sequence), and the number of lineages is unknown.
> * K-means is stochastic (random initialisation) and tends to produce spherical clusters of similar size, which is unsuitable for lineages that differ in size and divergence. Hierarchical clustering is deterministic and reproducible.
> * UPGMA assumes a roughly constant substitution rate (a molecular clock), which is reasonable for closely related viral orthologs, and is only a *distance-based approximation*. Publication-grade phylogenies need maximum-likelihood or Bayesian inference (e.g. IQ-TREE, BEAST) on nucleotide or codon alignments.

In [ ]:
# ----------------------- Distance matrices & dendrograms -----------------------
MAX_LEAVES = 60


def identity_distance_matrix(M):
    n = M.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            a, b = M[i], M[j]
            union = (a != "-") | (b != "-")
            match = (a == b) & (a != "-")
            D[i, j] = D[j, i] = 1.0 - match.sum() / max(int(union.sum()), 1)
    return D


TREES = {}
fig, axes = plt.subplots(1, len(ALN), figsize=(8.5 * len(ALN), 0.24 * MAX_LEAVES + 2.5), squeeze=False)
for ax, (prot, a) in zip(axes[0], ALN.items()):
    haps = a["haps"]
    sel = haps.sort_values("Freq", ascending=False).head(MAX_LEAVES).index.values
    sel = np.sort(sel)
    sub = haps.loc[sel].reset_index(drop=True)
    M = a["M"][sel]
    if len(sub) < 2:
        ax.text(0.5, 0.5, f"{prot}: fewer than 2 haplotypes -\nno tree can be built", ha="center", va="center")
        ax.axis("off")
        continue
    D = identity_distance_matrix(M)
    Z = hierarchy.linkage(squareform(D, checks=False), method="average")          # UPGMA
    coph, _ = hierarchy.cophenet(Z, squareform(D, checks=False))
    labels = [f"{h} (n={f})" for h, f in zip(sub.Haplotype, sub.Freq)]
    dn = hierarchy.dendrogram(Z, labels=labels, orientation="right", ax=ax, leaf_font_size=8,
                              color_threshold=0, above_threshold_color="#555555")
    cl_of = dict(zip(labels, sub.Clade))
    for t in ax.get_ymajorticklabels():
        t.set_color(CLADE_COLORS[cl_of[t.get_text()]])
        t.set_fontweight("bold")
    ax.set(title=f"{prot}: UPGMA dendrogram of {len(sub)} haplotypes (cophenetic r = {coph:.3f})",
           xlabel="Distance (1 - sequence identity)")
    ax.legend(handles=[Patch(color=CLADE_COLORS[c], label=c) for c in CLADES], loc="lower right", frameon=False)
    TREES[prot] = {"D": D, "Z": Z, "sub": sub, "coph": coph}
fig.tight_layout()
savefig(fig, "fig5_dendrograms.png")
plt.show()

# Do the two clades separate when the tree is cut into two clusters?
for prot, t in TREES.items():
    cl = hierarchy.fcluster(t["Z"], 2, criterion="maxclust")
    print(f"\n{prot}: cut into 2 clusters vs known clade (weighted by haplotype frequency)")
    tab = pd.crosstab(pd.Series(cl, name="Cluster"), pd.Series(t["sub"].Clade.values, name="Clade"),
                      values=t["sub"].Freq.values, aggfunc="sum").fillna(0).astype(int)
    display(tab)
    purity = tab.max(axis=1).sum() / tab.values.sum()
    print(f"   cluster purity with respect to clade: {purity:.1%}")
    inter = [t["D"][i, j] for i in range(len(t["sub"])) for j in range(i + 1, len(t["sub"]))
             if t["sub"].Clade[i] != t["sub"].Clade[j]]
    intra = [t["D"][i, j] for i in range(len(t["sub"])) for j in range(i + 1, len(t["sub"]))
             if t["sub"].Clade[i] == t["sub"].Clade[j]]
    if inter and intra:
        print(f"   mean distance between clades = {np.mean(inter):.4f}  vs  within clades = {np.mean(intra):.4f}")

In [ ]:
# ----------------------- Distance heatmaps (supporting view) -----------------------
if TREES:
    fig, axes = plt.subplots(1, len(TREES), figsize=(6.5 * len(TREES), 5.8), squeeze=False)
    for ax, (prot, t) in zip(axes[0], TREES.items()):
        order = hierarchy.leaves_list(t["Z"])
        Dm = t["D"][np.ix_(order, order)]
        sns.heatmap(Dm, ax=ax, cmap="viridis_r", square=True, cbar_kws={"label": "1 - identity", "shrink": 0.8},
                    xticklabels=False, yticklabels=False)
        # clade colour strip along the left edge
        for k, idx in enumerate(order):
            ax.add_patch(plt.Rectangle((-0.04 * len(order), k), 0.03 * len(order), 1, clip_on=False,
                                       color=CLADE_COLORS[t["sub"].Clade[idx]]))
        ax.set_title(f"{prot}: pairwise distance matrix (dendrogram order)")
    fig.tight_layout()
    savefig(fig, "fig6_distance_heatmaps.png")
    plt.show()

## 7. Biological Discussion, Limitations & Reproducibility Notes  *(KDD phase 5: Interpretation & Evaluation)*

The next cell prints the headline numbers from this run. The discussion that follows explains how to read them.

In [ ]:
# ----------------------- Auto-generated findings summary -----------------------
print("=" * 78)
print("HEADLINE FINDINGS FROM THIS RUN")
print("=" * 78)
print(f"Sequences after QC: {len(df)} (clade-classified: {len(df_cls)}; Unclassified: {len(df) - len(df_cls)})")
for prot in PROTEINS:
    if prot not in ALN:
        print(f"\n{prot}: no clade-classified data."); continue
    a, m = ALN[prot], ALN[prot]["mut"]
    n_i = int(((df_cls.Protein == prot) & (df_cls.Clade == "Clade I")).sum())
    n_ii = int(((df_cls.Protein == prot) & (df_cls.Clade == "Clade II")).sum())
    print(f"\n{prot}: {n_i} Clade I / {n_ii} Clade II sequences, {len(a['haps'])} haplotypes, length {a['M'].shape[1]} aa")
    if len(m):
        cd = m[m.Category == "Clade-defining (fixed)"]
        cb = m[m.Category == "Clade-biased (majority differs)"]
        print(f"  variable positions: {len(m)} | clade-defining: {len(cd)} | clade-biased: {len(cb)} | "
              f"intra-clade polymorphic: {int((m.Category == 'Intra-clade polymorphism').sum())}")
        if len(cd):
            print("  clade-defining substitutions (Clade I -> Clade II): " + ", ".join(
                f"{r.Substitution[0]}{r.Position}{r.Substitution[-1]} [{r.Substitution_type}]" for _, r in cd.iterrows()))
            print(f"  of which non-conservative: {int((cd.Substitution_type == 'Non-conservative').sum())}; "
                  f"charge-altering: {int((cd.Delta_charge.fillna(0) != 0).sum())}")
    cons = 100 * (a["H_all"] == 0).mean()
    print(f"  fully conserved columns (entropy = 0): {cons:.1f}% of positions")
    if (prot, "Clade I") in REFS and (prot, "Clade II") in REFS:
        (_, sI), (_, sII) = REFS[(prot, "Clade I")], REFS[(prot, "Clade II")]
        ga = g_al.align(sI, sII)[0]; mm, _, cc = aln_stats(ga, sI, sII)
        print(f"  reference-vs-reference global identity: {100 * mm / cc:.1f}%")
    ps = MWU[(MWU.Protein == prot) & (MWU.Level == "Haplotype-level") & MWU.p_value.notna()] if "p_value" in MWU else []
    if len(ps):
        best = ps.sort_values("p_value").iloc[0]
        print(f"  lowest haplotype-level p-value: {best.Property} p = {best.p_value:.3g} "
              f"(Bonferroni {best.p_Bonferroni:.3g})")
    if prot in TREES:
        print(f"  UPGMA cophenetic correlation: {TREES[prot]['coph']:.3f}")

### 7.1 Reading the results: cross-clade conservation

* **Conservation.** A29L and A35R are structural, essential virion proteins, so low per-position entropy and high Clade I vs Clade II identity are expected. The dendrogram and the cluster-purity table show whether the few substitutions are enough to separate the clades. Clean separation with a handful of clade-defining residues means the clades are distinguishable, but only at a few sites.
* **Mutation type.** Positions listed as *Clade-defining (fixed)* are the strongest candidates for clade-specific markers. Mostly **conservative** substitutions (BLOSUM62 > 0, small ΔKD, Δcharge = 0) suggest functional constraint is preserved. **Non-conservative or charge-altering** changes, especially in regions involved in heparan sulfate binding (A29L) or in the coiled-coil / glycosylated ectodomain (A35R), deserve closer structural and functional study.
* **Physicochemistry.** If the Mann-Whitney tests are not significant at the haplotype level after correction, surface charge (pI, net charge) and hydrophobicity (GRAVY) are effectively conserved across clades. A difference that is significant only at the *sequence* level, not the haplotype level, mostly reflects **sampling bias** (many identical outbreak sequences) and not a real biochemical shift.

### 7.2 Implications for pan-Mpox vaccines and diagnostics

* **Vaccines.** Orthopoxvirus A27/A29 and A33/A35 antigens are components of subunit and mRNA vaccine designs. High cross-clade conservation, with no clade-defining changes at known epitopes, supports a single antigen sequence protecting against both clades. Any clade-defining residue should be checked against mapped neutralizing epitopes (for example in IEDB) and, where relevant, against glycosylation sites before choosing a vaccine immunogen.
* **Diagnostics.** Conserved, low-entropy windows suit **pan-Mpox** assays (antibody or antigen tests, PCR targets at the matching gene region). Clade-defining positions can be used as **clade-discriminating** markers. Assay design should always be re-checked at nucleotide level, because synonymous variation is invisible in protein alignments.
* **Antigenic stability.** Instability index and aromaticity serve as coarse proxies for in-vitro stability of recombinant antigens. Large clade differences would argue for testing both variants in expression and stability studies.

### 7.3 Limitations

1. **Clade assignment is metadata-driven.** Protein records often lack country, date and isolate. Cameroon hosts both clades. Some records stay Unclassified, and rule-based calls can be wrong. Verify important calls against the linked nucleotide record or Nextstrain / NCBI Virus clade annotations.
2. **Sampling bias.** The 2022+ Clade IIb outbreak is heavily over-sequenced relative to Clade I. Haplotype-level statistics reduce but do not remove this.
3. **Annotation noise.** Records come from keyword searches and length filters, so a few unrelated proteins or partial or mis-annotated records may slip through (see the `Annot_Match` flag).
4. **Star alignment.** The multiple alignment is a reference-based approximation. Insertions relative to the reference are discarded, and numbering follows the Clade I reference. For indel-rich data, use MAFFT or MUSCLE.
5. **UPGMA.** The tree is distance-based and assumes a clock. It has no branch support values. Use a maximum-likelihood tree (IQ-TREE) with bootstrap support for any phylogenetic claim.
6. **Protein level only.** Synonymous changes, regulatory regions, structural context and experimental epitope data are out of scope.

### 7.4 Reproducibility: freezing the dataset

NCBI is a **living database**. New records are added, and annotations and versions change, so the same query can return different data next week. The Entrez API also rate-limits, throttles and times out. This notebook handles both problems:

* **Local-first ingestion.** The manually exported `mpox_a29l.gb` and `mpox_a35r.gb` are a fixed, citable snapshot that parses in seconds without network calls. Keep them (and the cache) under version control with the notebook.
* **Cache.** `mpox_raw_gb_cache.gb` freezes any API download, so a rerun does not hit NCBI again.
* **Timeouts and retries.** `socket.setdefaulttimeout(15)`, 50-ID chunks, 0.4 s pacing and 3 retries mean a stalled stream can never hang the kernel.
* **Manifest.** The next cell records input checksums, versions and counts.

In [ ]:
# ----------------------- Freeze outputs & write a reproducibility manifest -----------------------
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


df.drop(columns=["Description"]).to_csv("mpox_clean_dataset.csv", index=False)
HAP.to_csv("mpox_haplotypes.csv", index=False)
with open("mpox_haplotypes.fasta", "w") as fh:
    for _, r in HAP.iterrows():
        fh.write(f">{r.Haplotype}|{r.Protein}|{r.Clade}|n={r.Freq}\n{r.Sequence}\n")

inputs = [f for f in list(LOCAL_FILES.values()) + [CACHE_FILE] if os.path.isfile(f)]
manifest = {
    "project": "Comparative analysis of Mpox A29L and A35R (Clade I vs Clade II)",
    "author": "Cindy Ogutu, Strathmore University (cindy.ogutu@strathmore.edu)",
    "generated": datetime.now().isoformat(timespec="seconds"),
    "python": sys.version.split()[0], "biopython": Bio.__version__,
    "pandas": pd.__version__, "numpy": np.__version__,
    "input_files_sha256": {f: sha256(f) for f in inputs},
    "counts": {"raw_records": len(RAW_RECORDS), "after_qc": int(len(df)), "clade_classified": int(len(df_cls)),
               "haplotypes": int(len(HAP))},
    "length_filters": LENGTH_RANGE, "entrez_queries": [q for _, q in QUERIES],
}
json.dump(manifest, open("mpox_manifest.json", "w"), indent=2)
print(json.dumps(manifest, indent=2)[:1800])
print("\nFiles written: mpox_clean_dataset.csv, mpox_haplotypes.csv, mpox_haplotypes.fasta, "
      "mpox_mutation_summary.csv, mpox_physicochemical_features.csv, mpox_mannwhitney_results.csv, "
      "mpox_manifest.json, figures/*.png")

---
**End of notebook.** *Cindy Ogutu, Strathmore University, Computing and Bioinformatics.*
Tip: to re-run offline, keep `mpox_a29l.gb`, `mpox_a35r.gb` (or `mpox_raw_gb_cache.gb`) next to this notebook.